In [1]:
!git clone https://github.com/jinsuyoo/act.git
%cd act

Cloning into 'act'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 56 (delta 6), reused 47 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 323.54 KiB | 6.74 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/kaggle/working/act


In [2]:
!pip install -q pytorch-lightning==1.9.5
!pip install -q imageio scikit-image einops opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 11.6 MB/s eta 0:00:0000:0100:01


In [3]:
from pathlib import Path

p = Path("test.py")
t = p.read_text()

start = t.find("trainer = Trainer(")
end = t.find("    # test model")

new = """trainer = Trainer(
        accelerator="gpu",
        devices=1,
        default_root_dir=args.save_path,
        enable_progress_bar=False
    )

"""

t = t[:start] + new + t[end:]

p.write_text(t)

print("✅ test.py fixed")

✅ test.py fixed


In [4]:
from pathlib import Path

path = Path("src/utils/utils_image.py")
text = path.read_text()

if "import numpy as np" not in text:
    text = text.replace(
        "import math",
        "import math\nimport numpy as np\nfrom skimage.metrics import structural_similarity"
    )

if "def calc_ssim" not in text:
    text += """

def calc_ssim(sr, hr, scale):
    sr = sr.squeeze(0).permute(1, 2, 0).cpu().numpy()
    hr = hr.squeeze(0).permute(1, 2, 0).cpu().numpy()

    if scale > 0:
        sr = sr[scale:-scale, scale:-scale, :]
        hr = hr[scale:-scale, scale:-scale, :]

    sr = sr.astype(np.uint8)
    hr = hr.astype(np.uint8)

    return structural_similarity(
        hr,
        sr,
        channel_axis=2,
        data_range=255
    )
"""

path.write_text(text)
print("✅ utils_image.py patched")

✅ utils_image.py patched


In [5]:
from pathlib import Path

path = Path("src/models/act_module.py")
text = path.read_text()

if "calc_ssim" not in text:
    text = text.replace(
        "from src.utils.utils_image import quantize, calc_psnr",
        "from src.utils.utils_image import quantize, calc_psnr, calc_ssim"
    )

text = text.replace(
    "self.avg_psnr = []",
    "self.avg_psnr = []\n        self.avg_ssim = []"
)

text = text.replace(
    "psnr = calc_psnr(output, img_gt, self.scale, self.rgb_range)",
    "psnr = calc_psnr(output, img_gt, self.scale, self.rgb_range)\n        ssim = calc_ssim(output, img_gt, self.scale)"
)

text = text.replace(
    "self.text_logger.info(f'Filename: {filename[0]} | PSNR: {psnr:.3f}')",
    "self.text_logger.info(f'Filename: {filename[0]} | PSNR: {psnr:.3f} | SSIM: {ssim:.4f}')"
)

text = text.replace(
    "self.avg_psnr.append(psnr)",
    "self.avg_psnr.append(psnr)\n        self.avg_ssim.append(ssim)"
)

text = text.replace(
    "avg_psnr = np.mean(np.array(self.avg_psnr))\n        self.text_logger.info(f'Average PSNR: {avg_psnr:.3f}')",
    "avg_psnr = np.mean(np.array(self.avg_psnr))\n        avg_ssim = np.mean(np.array(self.avg_ssim))\n        self.text_logger.info(f'Average PSNR: {avg_psnr:.3f}')\n        self.text_logger.info(f'Average SSIM: {avg_ssim:.4f}')"
)

path.write_text(text)
print("✅ act_module.py patched")

✅ act_module.py patched


In [6]:
!grep -n "criterion" src/models/act_module.py

42:        self.criterion = nn.L1Loss()
353:        loss = self.criterion(output, img_gt)
365:        loss = self.criterion(output, img_gt).detach()


In [7]:
from pathlib import Path

path = Path("src/models/act_module.py")
text = path.read_text()

text = text.replace(
    "self.criterion = nn.L1Loss()",
    "self.criterion = nn.SmoothL1Loss(beta=1.0)"
)

path.write_text(text)
print("✅ Smooth L1 Loss enabled")

✅ Smooth L1 Loss enabled


In [8]:
!grep -R "class RCAB" -n src/models

src/models/act_net.py:40:class RCAB(nn.Module):


In [9]:
!sed -n '40,120p' src/models/act_net.py

class RCAB(nn.Module):
    def __init__(self,
                 conv,
                 n_feat,
                 kernel_size,
                 reduction,
                 bias=True,
                 bn=False,
                 act=nn.ReLU(True),
                 res_scale=1):

        super(RCAB, self).__init__()
        modules_body = []
        for i in range(2):
            modules_body.append(conv(n_feat, n_feat, kernel_size, bias=bias))
            if bn:
                modules_body.append(nn.BatchNorm2d(n_feat))
            if i == 0:
                modules_body.append(act)
        modules_body.append(CALayer(n_feat, reduction))
        self.body = nn.Sequential(*modules_body)
        self.res_scale = res_scale

    def forward(self, x):
        res = self.body(x)
        res += x
        return res


## Residual Group (RG)
class ResidualGroup(nn.Module):
    def __init__(self, conv, n_feat, kernel_size, reduction, n_resblocks):
        super(ResidualGroup, self).__init__()
      

In [10]:
!grep -n "class CALayer" -A40 src/models/act_net.py

20:class CALayer(nn.Module):
21-    def __init__(self, channel, reduction=16):
22-        super(CALayer, self).__init__()
23-        # global average pooling: feature --> point
24-        self.avg_pool = nn.AdaptiveAvgPool2d(1)
25-        # feature channel downscale and upscale --> channel weight
26-        self.conv_du = nn.Sequential(
27-            nn.Conv2d(channel, channel // reduction, 1, padding=0, bias=True),
28-            nn.ReLU(inplace=True),
29-            nn.Conv2d(channel // reduction, channel, 1, padding=0, bias=True),
30-            nn.Sigmoid(),
31-        )
32-
33-    def forward(self, x):
34-        y = self.avg_pool(x)
35-        y = self.conv_du(y)
36-        return x * y
37-
38-
39-## Residual Channel Attention Block (RCAB)
40-class RCAB(nn.Module):
41-    def __init__(self,
42-                 conv,
43-                 n_feat,
44-                 kernel_size,
45-                 reduction,
46-                 bias=True,
47-                 bn=False,
48-         

In [11]:
from pathlib import Path

path = Path("src/models/act_net.py")
text = path.read_text()

# ------------------------------------------------------------------
# Add SpatialAttention class after CALayer
# ------------------------------------------------------------------

if "class SpatialAttention" not in text:

    spatial = '''

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        padding = (kernel_size - 1) // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False,
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        y = self.sigmoid(y)

        return x * y

'''

    marker = "return x * y\n\n\n## Residual Channel Attention Block (RCAB)"
    text = text.replace(
        marker,
        "return x * y\n" + spatial + "\n## Residual Channel Attention Block (RCAB)"
    )

# ------------------------------------------------------------------
# Add Spatial Attention inside RCAB
# ------------------------------------------------------------------

old = "modules_body.append(CALayer(n_feat, reduction))"

new = """modules_body.append(CALayer(n_feat, reduction))
        modules_body.append(SpatialAttention())"""

text = text.replace(old, new)

path.write_text(text)

print("✅ Dual Attention patched successfully!")

✅ Dual Attention patched successfully!


In [12]:
!grep -n "SpatialAttention" src/models/act_net.py

39:class SpatialAttention(nn.Module):
41:        super(SpatialAttention, self).__init__()
87:        modules_body.append(SpatialAttention())


In [13]:
!grep -n "torch.cat" src/models/act_net.py

59:        y = torch.cat([avg_out, max_out], dim=1)
310:            x_tkn = torch.cat((x_tkn_a, x_tkn_b), -1)
320:            f = torch.cat((x, x_tkn), 1)


In [14]:
!grep -n "fusion" src/models/act_net.py

156:        n_fusionblocks = args.n_fusionblocks
159:        self.n_fusionblocks = args.n_fusionblocks
237:        self.fusion_block = nn.ModuleList([
243:            ) for _ in range(n_fusionblocks)
246:        self.fusion_mlp = nn.ModuleList([
252:            ) for _ in range(n_fusionblocks - 1)
255:        self.fusion_cnn = nn.ModuleList([
258:            ) for _ in range(n_fusionblocks - 1)
286:        for i in range(self.n_fusionblocks):
321:            f = f + self.fusion_block[i](f)
323:            if i != (self.n_fusionblocks - 1):
328:                x_tkn = self.fusion_mlp[i](x_tkn)+ x_tkn_res
330:                x = self.fusion_cnn[i](x) + x_res


In [15]:
!sed -n '280,340p' src/models/act_net.py


        x_tkn = F.unfold(x, self.token_size, stride=self.token_size)
        x_tkn = rearrange(x_tkn, 'b d t -> b t d')

        x_tkn = self.linear_encoding(x_tkn) + x_tkn

        for i in range(self.n_fusionblocks):
            x_tkn = self.mhsa_block[i][0](x_tkn) + x_tkn
            x_tkn = self.mhsa_block[i][1](x_tkn) + x_tkn

            x_tkn_a, x_tkn_b = torch.split(x_tkn, self.embedding_dim // 2, -1)

            x_tkn_b = rearrange(x_tkn_b, 'b t d -> b d t')
            x_tkn_b = F.fold(x_tkn_b, (h, w), self.token_size, stride=self.token_size)

            x_tkn_b = F.unfold(x_tkn_b, self.token_size * 2, stride=self.token_size)
            x_tkn_b = rearrange(x_tkn_b, 'b d t -> b t d')

            x_tkn_b = self.csta_block[i][0](x_tkn_b)
            _x_tkn_a, _x_tkn_b = x_tkn_a, x_tkn_b
            x_tkn_a = self.csta_block[i][1](x_tkn_a, _x_tkn_b) + x_tkn_a
            x_tkn_b = self.csta_block[i][2](x_tkn_b, _x_tkn_a) + x_tkn_b
            x_tkn_b = self.csta_block[i][3](

In [16]:
!grep -n "fusion_block" -A20 -B20 src/models/act_net.py

217-                    nn.Linear(embedding_dim // 2, embedding_dim * 2)
218-                ),
219-                # conventional FFN after the attention
220-                PreNorm(
221-                    embedding_dim,
222-                    FeedForward(embedding_dim, hidden_dim, dropout_rate)
223-                )
224-            ]) for _ in range(n_layers // 2)
225-        ])
226-
227-        # CNN Branch borrowed from RCAN
228-        modules_body = [
229-            ResidualGroup(conv, n_feats, 3, reduction, n_resblocks=n_resblocks)
230-            for _ in range(n_resgroups)
231-        ]
232-
233-        modules_body.append(conv(n_feats, n_feats, 3))
234-        self.cnn_branch = nn.Sequential(*modules_body)
235-
236-        # Fusion Blocks
237:        self.fusion_block = nn.ModuleList([
238-            nn.Sequential(
239-                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
240-                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
241-                FB(conv, 

In [17]:
from pathlib import Path
Path("src/models/adaptive_fusion.py").touch()

In [18]:
!ls src/models

act_module.py  act_net.py  adaptive_fusion.py  common.py  __init__.py


In [19]:
from pathlib import Path

code = '''
import torch
import torch.nn as nn

class AdaptiveFusionGate(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.gate(x)
        return x * w
'''

Path("src/models/adaptive_fusion.py").write_text(code)

print("✅ adaptive_fusion.py created")

✅ adaptive_fusion.py created


In [20]:
from pathlib import Path

path = Path("src/models/act_net.py")
text = path.read_text()

if "from src.models.adaptive_fusion import AdaptiveFusionGate" not in text:
    marker = "from einops import rearrange"
    text = text.replace(
        marker,
        marker + "\nfrom src.models.adaptive_fusion import AdaptiveFusionGate"
    )

path.write_text(text)

print("✅ AdaptiveFusionGate imported")

✅ AdaptiveFusionGate imported


In [21]:
from pathlib import Path

path = Path("src/models/act_net.py")
text = path.read_text()

old = """self.fusion_block = nn.ModuleList([
            nn.Sequential(
                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
                FB(conv, n_feats * 2, 1, act=nn.ReLU(True)),
            ) for _ in range(n_fusionblocks)
        ])"""

new = old + """

        self.adaptive_gate = nn.ModuleList([
            AdaptiveFusionGate(n_feats * 2)
            for _ in range(n_fusionblocks)
        ])"""

if "self.adaptive_gate" not in text:
    text = text.replace(old, new)

path.write_text(text)

print("✅ Adaptive gates added")

✅ Adaptive gates added


In [22]:
from pathlib import Path

path = Path("src/models/act_net.py")
text = path.read_text()

old = """f = f + self.fusion_block[i](f)"""

new = """f = f + self.fusion_block[i](f)
            f = self.adaptive_gate[i](f)"""

text = text.replace(old, new)

path.write_text(text)

print("✅ Adaptive Fusion integrated")

✅ Adaptive Fusion integrated


In [23]:
!grep -n "AdaptiveFusionGate" src/models/act_net.py

!grep -n "adaptive_gate" src/models/act_net.py

5:from src.models.adaptive_fusion import AdaptiveFusionGate
248:            AdaptiveFusionGate(n_feats * 2)
247:        self.adaptive_gate = nn.ModuleList([
328:            f = self.adaptive_gate[i](f)


In [24]:
!grep -R "load_state_dict" -n .

./src/models/__init__.py:27:            net.load_state_dict(torch.load(model_path))


In [25]:
from pathlib import Path

path = Path("src/models/__init__.py")
text = path.read_text()

text = text.replace(
    "net.load_state_dict(torch.load(model_path))",
    "net.load_state_dict(torch.load(model_path), strict=False)"
)

path.write_text(text)

print("✅ Pretrained weight loading patched (strict=False)")

✅ Pretrained weight loading patched (strict=False)


In [26]:
!grep -n "load_state_dict" src/models/__init__.py

27:            net.load_state_dict(torch.load(model_path), strict=False)


In [27]:
!python test.py \
--release \
--task sr \
--scale 2 \
--data_test Set5 \
--save_path sanity_check

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/sanity_check
Global seed set to 310
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
Missing logger folder: experiments/test/sanity_check/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going t

In [28]:
!python test.py \
--release \
--task sr \
--scale 2 \
--data_test Set5 \
--save_path sanity_check

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/sanity_check
Global seed set to 310
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get D

In [29]:
!find datasets/benchmark -maxdepth 2 -type d

find: ‘datasets/benchmark’: No such file or directory


In [30]:
!find datasets -maxdepth 3 -type d

datasets


In [31]:
!wget http://cv.snu.ac.kr/research/EDSR/benchmark.tar

--2026-07-10 10:27:17--  http://cv.snu.ac.kr/research/EDSR/benchmark.tar
Resolving cv.snu.ac.kr (cv.snu.ac.kr)... 147.46.67.83
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://cv.snu.ac.kr/00263465539/research/EDSR/benchmark.tar [following]
--2026-07-10 10:27:18--  http://cv.snu.ac.kr/00263465539/research/EDSR/benchmark.tar
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://cv.snu.ac.kr/research/EDSR/benchmark.tar [following]
--2026-07-10 10:27:18--  http://cv.snu.ac.kr/research/EDSR/benchmark.tar
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cv.snu.ac.kr/research/EDSR/benchmark.tar [following]
--2026-07-10 10:27:19--  https://cv.snu.ac.kr/research/EDSR/benchmark.tar
Connecting to cv.snu.ac.kr (cv.snu.ac

In [32]:
!tar -xf benchmark.tar
!mv benchmark datasets/

tar: benchmark.tar: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now
mv: cannot stat 'benchmark': No such file or directory


In [33]:
!find datasets/benchmark -maxdepth 2 -type d

find: ‘datasets/benchmark’: No such file or directory


In [34]:
!zip -r ACT_Code_Only.zip \
src \
configs \
train.py \
test.py \
README.md \
environments.yaml \
pretrained_weights \
-x "*.pyc" \
-x "__pycache__/*"

  adding: src/ (stored 0%)
  adding: src/data/ (stored 0%)
  adding: src/data/benchmark.py (deflated 57%)
  adding: src/data/datamodule.py (deflated 75%)
  adding: src/data/srdata.py (deflated 74%)
  adding: src/data/__init__.py (deflated 29%)
  adding: src/data/resize_right.py (deflated 69%)
  adding: src/data/imagenet.py (deflated 68%)
  adding: src/data/interp_method.py (deflated 67%)
  adding: src/data/__pycache__/ (stored 0%)
  adding: src/data/common.py (deflated 65%)
  adding: src/utils/ (stored 0%)
  adding: src/utils/utils_image.py (deflated 56%)
  adding: src/utils/__init__.py (stored 0%)
  adding: src/utils/utils_logger.py (deflated 51%)
  adding: src/utils/utils_saver.py (deflated 61%)
  adding: src/utils/__pycache__/ (stored 0%)
  adding: src/models/ (stored 0%)
  adding: src/models/act_net.py (deflated 80%)
  adding: src/models/__init__.py (deflated 60%)
  adding: src/models/act_module.py (deflated 79%)
  adding: src/models/adaptive_fusion.py (deflated 51%)
  adding: src/

In [35]:
!grep -R "max_epochs" -n configs src train.py

train.py:61:        max_epochs=args.epochs, 


In [36]:
!grep -R "lr" -n configs train.py

configs/option.py:65:    parser.add_argument('--lr', type=float, default=1e-4, 
grep: configs/__pycache__/option.cpython-312.pyc: binary file matches


In [37]:
!grep -R "data_train" -n configs train.py
!grep -R "DIV2K" -n configs src train.py

configs/option.py:97:    parser.add_argument('--data_train', type=str, default='ImageNet', 
grep: configs/__pycache__/option.cpython-312.pyc: binary file matches


In [38]:
!grep -R "epochs" -n configs/option.py

54:    parser.add_argument('--epochs', type=int, default=150, 
55:                        help='number of epochs to train')


In [39]:
!find datasets -maxdepth 3 -type d

datasets


In [40]:
!grep -R "create_model" -n train.py src
!grep -R "load_state_dict" -n train.py src

train.py:10:from src.models import create_model
train.py:37:    model = create_model(args, is_train=True)
src/models/__init__.py:11:def create_model(args, is_train=False):
grep: src/models/__pycache__/__init__.cpython-312.pyc: binary file matches
src/models/__init__.py:27:            net.load_state_dict(torch.load(model_path), strict=False)
grep: src/models/__pycache__/__init__.cpython-312.pyc: binary file matches


In [41]:
!find /kaggle/input -maxdepth 3 -type d

/kaggle/input


In [42]:
!ls src/data

benchmark.py  datamodule.py  __init__.py       __pycache__	srdata.py
common.py     imagenet.py    interp_method.py  resize_right.py


In [43]:
!grep -R "class DIV2K" -n src/data
!grep -R "class ImageNet" -n src/data

src/data/imagenet.py:59:class ImageNet():


In [44]:
!mkdir -p datasets/DIV2K
%cd datasets/DIV2K

!wget https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
!wget https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X2.zip

/kaggle/working/act/datasets/DIV2K
--2026-07-10 10:27:31--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3530603713 (3.3G) [application/zip]
Saving to: ‘DIV2K_train_HR.zip’

DIV2K_train_HR.zip  100%[===================>]   3.29G  33.7MB/s    in 1m 44s  

2026-07-10 10:29:15 (32.5 MB/s) - ‘DIV2K_train_HR.zip’ saved [3530603713/3530603713]

--2026-07-10 10:29:15--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X2.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 925390592 (883M) [application/zip]
Saving to

In [45]:
!unzip -q DIV2K_train_HR.zip
!unzip -q DIV2K_train_LR_bicubic_X2.zip

In [46]:
!find . -maxdepth 2 -type d

.
./DIV2K_train_LR_bicubic
./DIV2K_train_LR_bicubic/X2
./DIV2K_train_HR


In [47]:
%cd /kaggle/working/act

/kaggle/working/act


In [48]:
!ls src/data

benchmark.py  datamodule.py  __init__.py       __pycache__	srdata.py
common.py     imagenet.py    interp_method.py  resize_right.py


In [49]:
!grep -R "class ImageNet" -n src/data
!grep -R "__getitem__" -n src/data/imagenet.py

src/data/imagenet.py:59:class ImageNet():
98:    def __getitem__(self, idx):


In [50]:
!sed -n '1,220p' src/data/imagenet.py

import os
import random
from PIL import Image

import cv2
import torch
import numpy as np
import imageio

from src.data.resize_right import resize


def search(root, target='JPEG'):
    item_list = []
    items = os.listdir(root)
    for item in items:
        path = os.path.join(root, item)
        if os.path.isdir(path):
            item_list.extend(search(path, target))
        elif path.split('.')[-1] == target:
            item_list.append(path)
        elif path.split('/')[-1].startswith(target):
            item_list.append(path)
    return item_list


def get_patch_img(img, patch_size=96, scale=2):
    ih, iw = img.shape[:2]
    tp = scale * patch_size
    if (iw - tp) > -1 and (ih-tp) > 1:
        ix = random.randrange(0, iw-tp+1)
        iy = random.randrange(0, ih-tp+1)
        hr = img[iy:iy+tp, ix:ix+tp, :3]
    elif (iw - tp) > -1 and (ih - tp) <= -1:
        ix = random.randrange(0, iw-tp+1)
        hr = img[:, ix:ix+tp, :3]
        pil_img = Image.fromarray(hr).resize((

In [51]:
from pathlib import Path

path = Path("src/data/imagenet.py")
text = path.read_text()

old = """self.img_list = search(os.path.join(self.dataroot, 'imagenet', 'train'), 'JPEG')
        self.img_list.extend(search(os.path.join(self.dataroot, 'imagenet', 'val'), 'JPEG'))"""

new = """self.img_list = search(
            os.path.join(self.dataroot, 'DIV2K', 'DIV2K_train_HR'),
            'png'
        )"""

text = text.replace(old, new)

path.write_text(text)

print("✅ DIV2K dataloader patched")

✅ DIV2K dataloader patched


In [52]:
!grep -n "DIV2K" src/data/imagenet.py

66:            os.path.join(self.dataroot, 'DIV2K', 'DIV2K_train_HR'),


In [53]:
!find experiments -name "*.ckpt"

In [54]:
!python test.py --help | grep checkpoint

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
                        path to checkpoint


In [55]:
!python test.py --help | grep pre

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
               [--precision PRECISION] [--dir_data DIR_DATA]
  --release             store true for inference using pretrained weights
  --precision PRECISION
                        quality factor for image compression


In [56]:
!python test.py --help

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
usage: test.py [-h] [--release] [--num_workers NUM_WORKERS] [--cpu]
               [--gpus GPUS] [--seed SEED] [--task TASK] [--model MODEL]
               [--n_feats N_FEATS] [--act ACT] [--n_resgroups N_RESGROUPS]
               [--n_resblocks N_RESBLOCKS] [--reduction REDUCTION]
               [--token_size TOKEN_SIZE] [--n_heads N_HEADS]
               [--n_layers N_LAYERS] [--dropout_rate DROPOUT_RATE]
               [--expansion_ratio EXPANSION_RATIO]
               [--n_fusionblocks N_FUSIONBLOCKS] [--epochs EPOCHS]
               [--batch_size BATCH_SIZE] [--self_ensemble]
               [--crop_batch_size CROP_

In [57]:
!sed -n '1,120p' src/models/__init__.py

import os
import requests
from os import path as osp

import torch

from .act_net import ACT
from .act_module import ACTLitModule


def create_model(args, is_train=False):
    if is_train:
        return ACTLitModule(net=ACT(args), args=args)

    else: # test setting
        if args.release:
            model_path = f'pretrained_weights/act_{args.task}_x{args.scale}.pt'
            if not osp.exists(model_path):
                # download pretrained weight
                os.makedirs(osp.dirname(model_path), exist_ok=True)
                url = f'https://github.com/jinsuyoo/ACT/releases/download/v0.0.0/{osp.basename(model_path)}'
                r = requests.get(url, allow_redirects=True)
                print(f'Downloading pretrained weight: {model_path}')
                open(model_path, 'wb').write(r.content)
            
            net = ACT(args)
            net.load_state_dict(torch.load(model_path), strict=False)

            return ACTLitModule(net=net, args=args)

        el

In [58]:
!sed -n '1,120p' src/models/act_module.py

import os
from os import path as osp
from argparse import Namespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import make_grid
from pytorch_lightning import LightningModule
from einops import rearrange

from src.utils.utils_image import quantize, calc_psnr, calc_ssim


class ACTLitModule(LightningModule):
    def __init__(self, net: nn.Module, args: Namespace):
        super().__init__()

        self.args = args
        
        self.net = net

        self.task = args.task

        self.rgb_range = args.rgb_range
        self.scale = args.scale
        self.crop_batch_size = args.crop_batch_size 
        self.patch_size = args.patch_size
        self.self_ensemble = args.self_ensemble

        # optimization configs
        self.lr = args.lr
        self.step_size = args.decay
        self.gamma = args.gamma
        self.betas = args.betas
        self.eps = args.epsilon
        self.weight_decay = args.weight_decay


In [59]:
from pathlib import Path

path = Path("src/models/__init__.py")
text = path.read_text()

old = """        elif args.ckpt_path is not None:
            # use pretrained parameter
            assert osp.exists(args.ckpt_path), print(f'checkpoint not exists: {args.ckpt_path}')
            print(f'Loading checkpoint from: {args.ckpt_path}')

            return ACTLitModule.load_from_checkpoint(args.ckpt_path, args=args)"""

new = """        elif args.ckpt_path is not None:
            # use pretrained parameter
            assert osp.exists(args.ckpt_path), print(f'checkpoint not exists: {args.ckpt_path}')
            print(f'Loading checkpoint from: {args.ckpt_path}')

            net = ACT(args)

            return ACTLitModule.load_from_checkpoint(
                args.ckpt_path,
                net=net,
                args=args,
                strict=False
            )"""

text = text.replace(old, new)
path.write_text(text)

print("✅ Fixed checkpoint loading")

✅ Fixed checkpoint loading


In [60]:
!grep -n "load_from_checkpoint" src/models/__init__.py -A5

38:            return ACTLitModule.load_from_checkpoint(
39-                args.ckpt_path,
40-                net=net,
41-                args=args,
42-                strict=False
43-            )


In [61]:
!find datasets -maxdepth 3 -type d

datasets
datasets/DIV2K
datasets/DIV2K/DIV2K_train_LR_bicubic
datasets/DIV2K/DIV2K_train_LR_bicubic/X2
datasets/DIV2K/DIV2K_train_HR


In [62]:
!wget http://cv.snu.ac.kr/research/EDSR/benchmark.tar
!tar -xf benchmark.tar
!mv benchmark datasets/

--2026-07-10 10:30:54--  http://cv.snu.ac.kr/research/EDSR/benchmark.tar
Resolving cv.snu.ac.kr (cv.snu.ac.kr)... 147.46.67.83
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cv.snu.ac.kr/research/EDSR/benchmark.tar [following]
--2026-07-10 10:30:55--  https://cv.snu.ac.kr/research/EDSR/benchmark.tar
Connecting to cv.snu.ac.kr (cv.snu.ac.kr)|147.46.67.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 250112000 (239M) [application/x-tar]
Saving to: ‘benchmark.tar’

benchmark.tar       100%[===================>] 238.53M  4.37MB/s    in 57s     

2026-07-10 10:31:53 (4.17 MB/s) - ‘benchmark.tar’ saved [250112000/250112000]



In [63]:
!find datasets/benchmark -maxdepth 2 -type d

datasets/benchmark
datasets/benchmark/Urban100
datasets/benchmark/Urban100/LR_bicubic
datasets/benchmark/Urban100/HR
datasets/benchmark/Set5
datasets/benchmark/Set5/LR_bicubic
datasets/benchmark/Set5/HR
datasets/benchmark/B100
datasets/benchmark/B100/LR_bicubic
datasets/benchmark/B100/HR
datasets/benchmark/Set14
datasets/benchmark/Set14/LR_bicubic
datasets/benchmark/Set14/HR


In [64]:
!grep -R "load_state_dict" -n train.py src/models

src/models/__init__.py:27:            net.load_state_dict(torch.load(model_path), strict=False)
grep: src/models/__pycache__/__init__.cpython-312.pyc: binary file matches


In [65]:
!grep -R "release" -n train.py src/models

src/models/__init__.py:16:        if args.release:
src/models/__init__.py:21:                url = f'https://github.com/jinsuyoo/ACT/releases/download/v0.0.0/{osp.basename(model_path)}'
src/models/__init__.py:46:            raise ValueError('Need release option or checkpoint path')
grep: src/models/__pycache__/__init__.cpython-312.pyc: binary file matches


In [66]:
from pathlib import Path

path = Path("src/models/__init__.py")
text = path.read_text()

old = """    if is_train:
        return ACTLitModule(net=ACT(args), args=args)"""

new = """    if is_train:
        net = ACT(args)

        model_path = f'pretrained_weights/act_{args.task}_x{args.scale}.pt'

        if not osp.exists(model_path):
            os.makedirs(osp.dirname(model_path), exist_ok=True)

            url = f'https://github.com/jinsuyoo/ACT/releases/download/v0.0.0/{osp.basename(model_path)}'
            r = requests.get(url, allow_redirects=True)

            print(f'Downloading pretrained weight: {model_path}')
            open(model_path, 'wb').write(r.content)

        print(f'Loading pretrained ACT weights: {model_path}')

        net.load_state_dict(
            torch.load(model_path),
            strict=False
        )

        return ACTLitModule(net=net, args=args)"""

if old not in text:
    print("❌ Pattern not found.")
else:
    text = text.replace(old, new)
    path.write_text(text)
    print("✅ Training now loads pretrained ACT weights.")

✅ Training now loads pretrained ACT weights.


In [67]:
from pathlib import Path

path = Path("train.py")
text = path.read_text()

old = """trainer = Trainer(
        strategy='ddp',
        accelerator='gpu', 
        devices=args.gpus,
        logger=logger,
        callbacks=[checkpoint_callback],
        default_root_dir=args.save_path,
        max_epochs=args.epochs, 
        limit_train_batches=args.limit_train_batches,
        val_check_interval=args.val_check_interval,
        precision=args.precision,
        num_sanity_val_steps=args.num_sanity_val_steps,
        resume_from_checkpoint=None
    )

    # begin training
    trainer.fit(model=model, datamodule=datamodule)
"""

new = """trainer = Trainer(
        strategy='ddp',
        accelerator='gpu', 
        devices=args.gpus,
        logger=logger,
        callbacks=[checkpoint_callback],
        default_root_dir=args.save_path,
        max_epochs=args.epochs, 
        limit_train_batches=args.limit_train_batches,
        val_check_interval=args.val_check_interval,
        precision=args.precision,
        num_sanity_val_steps=args.num_sanity_val_steps,
    )

    # begin training
    trainer.fit(
        model=model,
        datamodule=datamodule,
        ckpt_path=args.ckpt_path if args.ckpt_path else None,
    )
"""

if old in text:
    text = text.replace(old, new)
    path.write_text(text)
    print("✅ train.py patched successfully.")
else:
    print("❌ Exact block not found. Showing surrounding lines:")
    for i, line in enumerate(text.splitlines(), 1):
        if "trainer = Trainer(" in line or "trainer.fit" in line:
            print(f"{i}: {line}")

✅ train.py patched successfully.


In [73]:
from pathlib import Path

p = Path("train.py")
text = p.read_text()

text = text.replace(
    "val_check_interval=args.val_check_interval,",
    "check_val_every_n_epoch=10,"
)

text = text.replace(
    "num_sanity_val_steps=args.num_sanity_val_steps,",
    "num_sanity_val_steps=0,"
)

p.write_text(text)
print("✅ Patched train.py")

✅ Patched train.py


In [74]:
!python train.py \
--epochs 150 \
--lr 1e-5 \
--batch_size 4 \
--dir_data datasets \
--data_train ImageNet \
--scale 2 \
--save_path act_improved_finetune_150_lr1e5 \
--num_workers 4

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/train/act_improved_finetune_150_lr1e5
Global seed set to 310
Loading pretrained ACT weights: pretrained_weights/act_sr_x2.pt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
[rank: 0] Global seed set to 310
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_b

In [75]:
!ls -lh experiments/train/act_improved_finetune_150_lr1e5/checkpoint

total 1.1G
-rw-r--r-- 1 root root 528M Jul 10 11:45 best-epoch69-val_psnr34.52.ckpt
-rw-r--r-- 1 root root 528M Jul 10 13:07 last.ckpt


In [76]:
!python test.py \
--dir_data datasets \
--data_test Set5 \
--scale 2 \
--ckpt_path experiments/train/act_improved_finetune_150_lr1e5/checkpoint/last.ckpt \
--save_path experiments/test/set5_150epoch

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/experiments/test/set5_150epoch
Global seed set to 310
Loading checkpoint from: experiments/train/act_improved_finetune_150_lr1e5/checkpoint/last.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
Missing logger folder: experiments/test/experiments/test/set5_150epoch/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker 

In [70]:
!python test.py \
--dir_data datasets \
--data_test Set14 \
--scale 2 \
--ckpt_path experiments/train/act_improved_finetune_150_lr1e5/checkpoint/last.ckpt \
--save_path experiments/test/set14_80epoch

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/experiments/test/set14_80epoch
Global seed set to 310
checkpoint not exists: experiments/train/act_improved_finetune_150_lr1e5/checkpoint/last.ckpt
Traceback (most recent call last):
  File "/kaggle/working/act/test.py", line 53, in <module>
    main()
  File "/kaggle/working/act/test.py", line 34, in main
    model = create_model(args)
            ^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/act/src/models/__init__.py", line 53, in create_model
    assert osp.exists(args.ckpt_path), print(f'checkpoint not exists: {args.ckpt_path}')
           ^^^^^^^^^^^^^^^^^^^^^^^

In [79]:
!python test.py \
--dir_data datasets \
--data_test Set14 \
--scale 2 \
--ckpt_path experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt \
--save_path experiments/test/set14_best

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/experiments/test/set14_best
Global seed set to 310
Loading checkpoint from: experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
Missing logger folder: experiments/test/experiments/test/set14_best/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will c

In [81]:
!grep -R "data_test" -n src/data

src/data/datamodule.py:14:        self.data_test = args.data_test
src/data/datamodule.py:36:        if self.data_test in ['Set5', 'Set14', 'B100', 'Urban100', 'Manga109']:
src/data/datamodule.py:39:                self.args, train=False, name=self.data_test
src/data/datamodule.py:44:                f'Incorrect test dataset [{self.data_test}] is given'
src/data/datamodule.py:57:        if self.data_test in ['Set5', 'Set14', 'B100', 'Urban100', 'Manga109']:
src/data/datamodule.py:60:                self.args, train=False, name=self.data_test
src/data/datamodule.py:65:                f'Incorrect test dataset [{self.data_test}] is given'
grep: src/data/__pycache__/datamodule.cpython-312.pyc: binary file matches


In [82]:
!python test.py \
--dir_data datasets \
--data_test B100 \
--scale 2 \
--ckpt_path experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt \
--save_path experiments/test/b100_best

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/experiments/test/b100_best
Global seed set to 310
Loading checkpoint from: experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
Missing logger folder: experiments/test/experiments/test/b100_best/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will cre

In [83]:
!python test.py \
--dir_data datasets \
--data_test Urban100 \
--scale 2 \
--ckpt_path experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt \
--save_path experiments/test/urban100_best

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Experimental results will be saved at: experiments/test/experiments/test/urban100_best
Global seed set to 310
Loading checkpoint from: experiments/train/act_improved_finetune_150_lr1e5/checkpoint/best-epoch69-val_psnr34.52.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Start test!
Missing logger folder: experiments/test/experiments/test/urban100_best/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader 